# Импорт библиотек и настройка среды

In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_colwidth', None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Удобства для навигации

In [3]:
navigation = dict()

# Вспомогательные функции

## Функция для составления таблицы количественных характеристик pandas DataFrame

In [4]:
def make_num_charactericstic_table(
    data: pd.DataFrame,
    percentiles: list[float] = [0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.995]
) -> pd.DataFrame:
    return data.drop(columns="target", errors="ignore").describe(percentiles=percentiles).T

In [5]:
navigation["функция для составления таблицы количественных характеристик"] = "make_num_characteristic_table"

## Функция для составления отчёта по количеству, rate и percent значения какого-либо признака

In [6]:
def make_feature_num_analysis(
    data: pd.DataFrame | pd.Series,
    feature_name: str
) -> pd.Series:
    analysis = (
        data
        .agg(["sum", "mean"])
        .T
        .rename(index={"sum": f"{feature_name}_count", "mean": f"{feature_name}_rate"})
    )
    analysis[f"{feature_name}_percent"] = analysis[f"{feature_name}_rate"] * 100

    return analysis

In [7]:
navigation["функция для составления отчёта по количеству, rate и percent значения какого-либо признака"] = "make_feature_num_analysis"

## Функция для проведения z-тестов

In [8]:
def make_z_test(
    data: pd.DataFrame,
    mask: pd.DataFrame,
    alpha: float = 0.05
) -> bool:
    count = [
        data.loc[mask, "target"].sum(),
        data.loc[~mask, "target"].sum()
    ]

    nobs = [
        mask.sum(),
        (~mask).sum()
    ]

    z_stat, p_value = proportions_ztest(count, nobs)

    return p_value < alpha

# Загрузка датасета

Загрузка датасета.

In [9]:
data = pd.read_csv("../data/cs-training.csv")
data = data.drop(columns=["Unnamed: 0"])

In [10]:
navigation["переменная датасета"] = "data"

Улучшенное наименование столбцов.

In [11]:
data = data.rename(columns={
    "SeriousDlqin2yrs": "target",
    "RevolvingUtilizationOfUnsecuredLines": "revolving_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "num_30_59_days_late",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "num_open_credit_lines",
    "NumberOfTimes90DaysLate": "num_90_days_late",
    "NumberRealEstateLoansOrLines": "num_real_estate_loans",
    "NumberOfTime60-89DaysPastDueNotWorse": "num_60_89_days_late",
    "NumberOfDependents": "num_dependents"
})

# Краткая характеристика датасета

Словарь данных.

In [12]:
data_dictionary = pd.DataFrame({
    "column": data.columns,
    "description": [
        "Наличие у клиента просрочки 90 и более дней или более серьёзное нарушение платёжной дисциплины.",
        "Общий баланс по кредитным картам и личным кредитным линиям, кроме недвижимости и долгов в рассрочку, например автокредитов, делённый на сумму кредитных лимитов.",
        "Возраст заёмщика в годах.",
        "Количество раз, когда заёмщик имел просрочку 30–59 дней, но не хуже, за последние 2 года.",
        "Ежемесячные выплаты по долгам, алименты и расходы на проживание, делённые на ежемесячный валовый доход.",
        "Ежемесячный доход.",
        "Количество открытых кредитов, например автокредит или ипотека, и кредитных линий, например кредитных карт.",
        "Количество раз, когда заёмщик имел просрочку 90 дней или более.",
        "Количество ипотечных и других кредитов, связанных с недвижимостью, включая кредитные линии под залог жилья.",
        "Количество раз, когда заёмщик имел просрочку 60–89 дней, но не хуже, за последние 2 года.",
        "Количество иждивенцев в семье, не включая самого заёмщика: супруг/супруга, дети и т. д."
    ],
    "types": [
        "binary",
        "percent",
        "integer",
        "count",
        "percent",
        "float",
        "count",
        "count",
        "count",
        "count",
        "count"
    ],
    "role": [
        "target",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature",
        "feature"
    ]
})

In [13]:
navigation["словарь данных"] = "data_dictionary"

In [14]:
data_dictionary

,column,description,types,role
0,target,Наличие у клиента просрочки 90 и более дней или более серьёзное нарушение платёжной дисциплины.,binary,target
1,revolving_utilization,"Общий баланс по кредитным картам и личным кредитным линиям, кроме недвижимости и долгов в рассрочку, например автокредитов, делённый на сумму кредитных лимитов.",percent,feature
2,age,Возраст заёмщика в годах.,integer,feature
3,num_30_59_days_late,"Количество раз, когда заёмщик имел просрочку 30–59 дней, но не хуже, за последние 2 года.",count,feature
4,debt_ratio,"Ежемесячные выплаты по долгам, алименты и расходы на проживание, делённые на ежемесячный валовый доход.",percent,feature
5,monthly_income,Ежемесячный доход.,float,feature
6,num_open_credit_lines,"Количество открытых кредитов, например автокредит или ипотека, и кредитных линий, например кредитных карт.",count,feature
7,num_90_days_late,"Количество раз, когда заёмщик имел просрочку 90 дней или более.",count,feature
8,num_real_estate_loans,"Количество ипотечных и других кредитов, связанных с недвижимостью, включая кредитные линии под залог жилья.",count,feature
9,num_60_89_days_late,"Количество раз, когда заёмщик имел просрочку 60–89 дней, но не хуже, за последние 2 года.",count,feature


**Временная схема:**

Важно отметить, какую временную схему имеют данные. Изобразить её можно следующим образом:

<div align="center">
    <img src="../../docs/images/time_scheme.png" width="700">
</div>

Скоринг происходит в момент $T_{0}$, признаки относятся к прошлому (Период никак не ограничен, кроме как в признаках `num_30_59_days_late` и `num_60_89_days_late`. Данные столбцы отражают информацию не ранее чем за 2 года до $T_{0}$). Целевая переменная (`target`) относится к будущему, следующие 2 года после $T_{0}$.

Таким образом, явных признаков temporal leakage не обнаружено.

Объём датасета

In [15]:
data.shape[0]

150000

In [16]:
navigation["размерность датасета"] = f"Количество объектов = {data.shape[0]}; количество признаков (включая target) = {data.shape[1]}"

In [17]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   target                 150000 non-null  int64  
 1   revolving_utilization  150000 non-null  float64
 2   age                    150000 non-null  int64  
 3   num_30_59_days_late    150000 non-null  int64  
 4   debt_ratio             150000 non-null  float64
 5   monthly_income         120269 non-null  float64
 6   num_open_credit_lines  150000 non-null  int64  
 7   num_90_days_late       150000 non-null  int64  
 8   num_real_estate_loans  150000 non-null  int64  
 9   num_60_89_days_late    150000 non-null  int64  
 10  num_dependents         146076 non-null  float64
dtypes: float64(4), int64(7)
memory usage: 12.6 MB


Как данные выглядят.

In [18]:
data.sample(5)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
4748,0,0.3126,44,0,0.4424,"9,500.0000",10,0,2,0,0.0000
109568,0,0.1199,51,0,0.6442,"6,500.0000",12,0,2,0,2.0000
37075,0,0.9202,41,0,0.4601,"6,620.0000",8,0,1,0,1.0000
110969,0,0.0079,65,0,0.1674,"3,583.0000",20,0,1,0,0.0000
59091,0,0.0833,38,0,0.0145,"5,115.0000",3,0,0,0,0.0000


Наличие пропусков.

In [19]:
report_on_missings = data.isna().sum()

In [20]:
navigation["отчёт по пропускам"] = "report_on_missings"

Наличие дубликатов.

In [21]:
num_duplicated = data.duplicated().sum()

In [22]:
navigation["количество дубликатов"] = "num_duplicated"

Общий bad rate.

In [23]:
overall_bad_rate = round(data["target"].mean(), 4)

In [24]:
overall_bad_rate

np.float64(0.0668)

In [25]:
navigation["общий bad rate"] = "overall_bad_rate"

Доверительный интервал bad rate.

In [26]:
ci_low_overall_bad_rate, ci_high_overall_bad_rate = proportion_confint(
    count=data["target"].sum(),
    nobs=data.shape[0],
    alpha=0.05,
    method="wilson"
)
ci_overall_bad_rate = (round(ci_low_overall_bad_rate, 4), round(ci_high_overall_bad_rate, 4))

In [27]:
round(ci_low_overall_bad_rate, 4), round(ci_high_overall_bad_rate, 4)

(0.0656, 0.0681)

In [28]:
navigation["доверительный интервал общего bad rate"] = "ci_overall_bad_rate"

**Вывод по блоку:**
1. Предварительно датасет насчитывает порядка 150 тысяч заёмщиков (есть дубликаты);
2. Каждый заёмщик характеризуется 10 признаками;
3. По описанию данных признаки рассматриваются как информация, доступная на момент $T_{0}$, а `target` относится к следующим двум годам. Поэтому явных признаков temporal leakage не обнаружено;
4. В данных присутствуют пропуски в признаках `monthly_income` и `num_dependents`, их нужно будет тщательно проанализировать;
5. bad rate по всему набору данных составляет порядка 6.684%. Классы сильно несбалансированы, нужно это будет учитывать.

# Data quality

## Возможные риски, связанные с схемой данных

**Сразу обсужу проблемы и риски связанные с самой схемой данных, на которые нужно будет делать поправки при анализе:**

1. `revolving_utilization`:

    Поскольку признак является отношением задолженности к кредитному лимиту, его экстремальные значения могут возникать как из-за действительно высокой утилизации, так и вследствие особенностей расчёта при очень малом/нулевом лимите или ошибок данных. Исходная документация не описывает обработку таких случаев, поэтому экстремальные значения требуют отдельной проверки
2. `debt_ratio`:

    1. Во-первых, данный признак имеет аналогичную проблему с делением на ноль и возможность получить неадекватное значение (так как данный признак является отношением).
    2. Во-вторых, `debt_ratio` агрегирует несколько видов обязательных платежей, поэтому одинаковое значение показателя может соответствовать разной экономической структуре расходов. Из-за отсутствия отдельных компонент детальная интерпретация признака ограничена.

Данные потенциальные проблемы сильно снижают предсказуемость значений данных признаков (особенно `debt_ratio`), что может существенно сказать на качестве анализа. **Поэтому, ответственным за схему данных и данные признаки рекомендуется:**
1. Сформировать чёткие соглашения, что будет делаться в случае, когда знаменатель не известен или равен нулю;
2. Разделить признак `debt_ratio` по смыслу на несколько признаков (например, на `debt_rate`, `alimony_rate` и `rent_rate`) для повышения интерпретируемости.

## Резюме

Переменная резюме.

In [29]:
quality_resume = pd.DataFrame(columns=["feature", "limitations", "summary", "actions"])

In [30]:
navigation["резюме анализа качества данных"] = 'quality_resume.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})'

Резюме (выполнить после прогона всего ноутбука).

In [ ]:
# quality_resume.style.set_properties(**{
#     "text-align": "left",
#     "white-space": "pre-wrap"
# })

,feature,limitations,summary,actions
0,duplicate,Нет никаких ограничений.,"Дубликаты пусть и выглядят как пустные данные (что выражается в количественных характеристиках их представителей), однако сами объекты не обладают уникальными идентификаторами, а их большее количество в принципе делают не удивительными, дубликаты по всем значениям. Таким образом, вполне возможно, что это дубликаты не по объектам, а по значениям.",Ничего не предпринимать.
1,revolving_utilization,1. признак не может принимать отрицательных значений; 2. признак не должен принимать значения сильно выше 1; 3. положителельные значения признака должны сопровождаться положительными значениями `num_open_credit_lines`.,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. экстремально большие значения, вероятнее всего, объясняются сменой смысла признака: из отношения к абсолютным значениям; 3. в рамках текущей документации сделать вывод о ситуации, когда `revolving_utilization` положителен, а `num_open_credit_lines` = 0, не представляется возможным",1. сохранить исходное значение; отдельно отметить экстремальные значения; способ обработки определить после анализа их связи с риском.
2,age,1. признак не может принимать отрицательных значений; 2. признак не может принимать значения меньше 18 лет (возраст совершеннолетия); 3. признак не должен принимать значения сильно выше 90.,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. есть один представитель с возрастом 0 лет - это явно ошибка; 3. сделать вывод об ошибочности представителей с экстремальными значениями в рамках текущей документации невозможно, считаем их нормальными.",1. сделать флаг об ошибке в признаке (возраст ниже 18 лет).
3,num_30_59_days_late / num_60_89_days_late / num_90_days_late,"1. признаки не могут принимать отрицательных значений; 2. признаки не могут принимать слишком большие значения (порядка 50, сотен и так далее).","1. признаки не содержат ошибок, связанных с отрицательными значениями; 2. признаки не содержат ненормально больших значений, кроме 96 и 98; 3. значения 96 и 98 являются системными кодами.",1. сделать флаги о системных кодах 96 и 98.
4,debt_ratio,1. признак не может принимать отрицательных значений; 2. признак не должен принимать значения сильно выше 1.,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. экстремальные значения имеют распределение, несовместимое с буквальной интерпретацией показателя как обычного ratio. Возможными объяснениями являются особенности расчёта показателя при малом/нулевом знаменателе, изменение семантики поля или ошибки данных. Исходная документация не позволяет установить точную причину.",1. сохранить исходное значение; отдельно отметить экстремальные значения; способ обработки определить после анализа их связи с риском.
5,monthly_income,"1. признак не может принимать отрицательных значений; 2. признак не может принимать совсем больших значений, порядка десятков и сотен миллионов.","1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не содержит ошибок, связанных с экстремальными значениями.",Ничего не предпринимать.
6,num_open_credit_lines,1. признак не может принимать отрицательных значений; 2. признак не может принимать сильно больших значений (порядка сотен).,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не содержит ошибок, связанных с экстремальными значениями.",Ничего не предпринимать.
7,num_real_estate_loans,1. признак не может принимать отрицательных значений; 2. признак не может принимать сильно больших значений (порядка сотен).,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не содержит ошибок, связанных с экстремальными значениями.",Ничего не предпринимать.
8,num_dependents,1. признак не может принимать отрицательных значений; 2. признак не может принимать сильно больших значений (порядка сотен).,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не 

## Анализ дуликатов

Количество дубликатов.

In [32]:
duplicate_num_characteristics = make_feature_num_analysis(
    data=data.duplicated(),
    feature_name="duplicate"
)

In [33]:
duplicate_num_characteristics

duplicate_count     609.0000
duplicate_rate        0.0041
duplicate_percent     0.4060
dtype: float64

In [34]:
navigation["количественные характеристики дубликатов"] = "duplicate_num_characteristics"

Посмотрим на представителей дубликатов.

In [35]:
data[data.duplicated()].sample(20)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
56196,0,1.0000,24,0,0.0000,NaN,0,0,0,0,NaN
68710,0,0.0000,81,0,0.0000,NaN,3,0,0,0,0.0000
85222,0,1.0000,22,0,24.0000,NaN,0,0,0,0,NaN
28954,0,1.0000,50,0,0.0000,NaN,0,0,0,0,0.0000
88132,0,0.0000,22,0,0.0000,NaN,1,0,0,0,0.0000
127724,0,0.0000,21,0,0.0000,NaN,1,0,0,0,0.0000
144153,0,1.0000,28,0,0.0000,"2,200.0000",0,0,0,0,0.0000
64216,0,1.0000,79,0,0.0000,NaN,1,0,0,0,0.0000
109194,0,1.0000,22,98,0.0000,NaN,0,98,0,98,0.0000
100712,1,1.0000,25,98,0.0000,NaN,0,98,0,98,0.0000


Дубликаты выглядят как какие-то пустые, незавершённые или неудачные попытки по подлюкчению клиента. Визуально об этом говорит:
1. `revolving_utilization` содержит только 1 и 0;
2. столбцы о количествах просрочек преимущественно содержат 0;
3. `debt_ratio` содержит преимущественно 0;
4. `monthly_income` содержит много пропущенных значений;
5. столбцы `num_open_credit_lines` и `num_real_estate_loans` преимущественно содержат нули;
6. `num_dependents` преимущественно содержит либо нули, либо пропуски.

Проверим данные наблюдения более тщательно.

In [36]:
duplicate_characteristics =\
    make_num_charactericstic_table(data[data.duplicated()])

overall_characteristics =\
    make_num_charactericstic_table(data)

In [37]:
duplicate_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,609.0000,0.4926,0.5004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
age,609.0000,49.5895,23.3218,21.0000,21.0000,21.0000,22.0000,22.0000,24.0000,51.0000,70.0000,82.0000,86.0000,91.0000,91.0000,99.0000
num_30_59_days_late,609.0000,7.0805,25.3931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,98.0000,98.0000,98.0000,98.0000
debt_ratio,609.0000,1.8100,29.8002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.8400,21.4000,520.0000
monthly_income,99.0000,750.2929,652.6119,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,820.0000,898.0000,"1,000.0000","2,210.0000","3,206.0000","3,353.0000","3,500.0000"
num_open_credit_lines,609.0000,1.5517,1.5710,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,4.0000,6.9200,7.9600,10.0000
num_90_days_late,609.0000,7.0985,25.3884,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,98.0000,98.0000,98.0000,98.0000
num_real_estate_loans,609.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
num_60_89_days_late,609.0000,7.0805,25.3931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,98.0000,98.0000,98.0000,98.0000
num_dependents,513.0000,0.0078,0.1080,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4400,2.0000


In [38]:
navigation["количественные характеристики дубликатов"] = "duplicate_characteristics"

In [39]:
overall_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"150,000.0000",6.0484,249.7554,0.0000,0.0000,0.0000,0.0000,0.0030,0.0299,0.1542,0.5590,0.9813,1.0000,1.0930,1.3663,"50,708.0000"
age,"150,000.0000",52.2952,14.7719,0.0000,23.0000,24.0000,29.0000,33.0000,41.0000,52.0000,63.0000,72.0000,78.0000,87.0000,89.0000,109.0000
num_30_59_days_late,"150,000.0000",0.4210,4.1928,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,5.0000,98.0000
debt_ratio,"150,000.0000",353.0051,"2,037.8185",0.0000,0.0000,0.0000,0.0043,0.0309,0.1751,0.3665,0.8683,"1,267.0000","2,449.0000","4,979.0400","6,186.0100","329,664.0000"
monthly_income,"120,269.0000","6,670.2212","14,384.6742",0.0000,0.0000,0.0000,"1,300.0000","2,005.0000","3,400.0000","5,400.0000","8,249.0000","11,666.0000","14,587.6000","25,000.0000","35,000.0000","3,008,750.0000"
num_open_credit_lines,"150,000.0000",8.4528,5.1460,0.0000,0.0000,0.0000,2.0000,3.0000,5.0000,8.0000,11.0000,15.0000,18.0000,24.0000,27.0000,58.0000
num_90_days_late,"150,000.0000",0.2660,4.1693,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,4.0000,98.0000
num_real_estate_loans,"150,000.0000",1.0182,1.1298,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,3.0000,4.0000,6.0000,54.0000
num_60_89_days_late,"150,000.0000",0.2404,4.1552,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,98.0000
num_dependents,"146,076.0000",0.7572,1.1151,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,4.0000,5.0000,20.0000


In [40]:
navigation["количественные характеристики датасета"] = "overall_characteristics"

В целом предположения верны, дубликаты в основном содержат пропуски и нули.

Проверим, ещё bad rate дубликатов и сравним его с bad rate всего датасета.

In [41]:
duplicate_bad_rate = round(data[data.duplicated()]["target"].mean(), 4)

In [42]:
navigation["bad rate дубликатов"] = "duplicate_bad_rate"

In [43]:
duplicate_bad_rate

np.float64(0.0279)

In [44]:
ci_low_duplicate_bad_rate, ci_high_duplicate_bad_rate = proportion_confint(
    count=data[data.duplicated]["target"].sum(),
    nobs=data[data.duplicated()].shape[0],
    alpha=0.05,
    method="wilson"
)
ci_duplicate_bad_rate = (round(ci_low_duplicate_bad_rate, 4), round(ci_high_duplicate_bad_rate, 4))

In [45]:
navigation["доверительный интервал bad rate дубликатов"] = "ci_duplicate_bad_rate"

In [46]:
ci_duplicate_bad_rate

(0.0175, 0.0442)

In [47]:
ci_overall_bad_rate

(0.0656, 0.0681)

bad rate отличается, проверим значимость такого отличия.

In [48]:
mask = data.duplicated()

make_z_test(data=data, mask=mask, alpha=0.05)

np.True_

При уровне значимости 0.05 данное отличие является статистически значимым.

Проверим ещё насколько много не полных дубликатов (если снизить количество признаков).

In [49]:
temp = data.drop(columns="target")
columns = temp.columns.to_list()

min_num_duplicated = temp.drop(columns=columns[0]).duplicated().sum()
max_num_duplicated = temp.drop(columns=columns[0]).duplicated().sum()
avg_num_duplicated = 0.0
for column in columns:
    current_num_duplicated = temp.drop(columns=column).duplicated().sum()

    if min_num_duplicated > current_num_duplicated:
        min_num_duplicated = current_num_duplicated

    if max_num_duplicated < current_num_duplicated:
        max_num_duplicated = current_num_duplicated

    avg_num_duplicated += current_num_duplicated

avg_num_duplicated /= len(temp.columns)

In [50]:
min_num_duplicated, avg_num_duplicated, max_num_duplicated

(np.int64(646), np.float64(1193.5), np.int64(2164))

In [51]:
navigation["минимальное количество дубликатов при использоании не полного набора признаков (на 1 меньше)"] = "min_num_duplicated"
navigation["среднее количество дубликатов при использоании не полного набора признаков (на 1 меньше)"] = "avg_num_duplicated"
navigation["максимальное количество дубликатов при использоании не полного набора признаков (на 1 меньше)"] = "max_num_duplicated"

Видно, что количество дубликатов в среднем повысилось почти в двое, а в максимальном случае, ппримерно втрое. Всё это наталкивает на мысль о том, что данные дубликаты не являются чем-то особенным или ошибочным, объектов достаточно много, сами объекты не имеют уникальных идентификаторов, поэтому нельзя сделать выводы, что данные дубликаты - это одни и те же объекты, а не разные объекты с совпавшими значениями.

**Вывод по дубликатам:**
1. Дубликаты выглядят как незавершённые или пустые данные, что выражается в количественных характеристиках дубликатов;
2. bad rate дубликатов отличается статистически значимо от bad rate не дубликатов;
3. При решении о добросовестности заёмщика нельзя опираться на наличие в базе данных дубликата его профиля;
4. Так как объекты не имеют уникальных идентификаторов, то дубликаты могут быть разными объектами просто с совпавшими значениями.

Таким образом, ничего с дубликатами делать не стоит.

In [52]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "duplicate",
    "Нет никаких ограничений.",
    "Дубликаты пусть и выглядят как пустные данные (что выражается в количественных характеристиках их представителей), "
    "однако сами объекты не обладают уникальными идентификаторами, а их большее количество в принципе делают не удивительными, "
    "дубликаты по всем значениям. Таким образом, вполне возможно, что это дубликаты не по объектам, а по значениям.",
    "Ничего не предпринимать."
]

## Анализ качества признака `revolving_utilization`

**revolving_utilization** - общий баланс по кредитным картам и личным кредитным линиям, кроме недвижимости и долгов в рассрочку, например автокредитов, делённый на сумму кредитных лимитов.

Исходя из определения признака:
1. Признак не может принимать отрицательные значения (так как никаких особых случаев описано не было);
2. Признак не должен сильно превышать 1, так как пусть банки и позволяют взять больше лимита в некоторых случаях (overdruft), но всё равно не в несколько раз;
3. Признак должен коррелировать с `num_open_credit_lines`, так как кажется, невозможным наличие кридитных лимитов и долгов по кредитам без открытых кредитных линий.

**Проверим это:**

**Наличие отрицательных значений.**

In [53]:
(data["revolving_utilization"] < 0).sum()

np.int64(0)

Отрицательных значений нет.

**Наличие экстремальных значений.**

Посмотрим на общее распределение значений признака.

In [54]:
overall_characteristics.loc["revolving_utilization"]

count   150,000.0000
mean          6.0484
std         249.7554
min           0.0000
0.5%          0.0000
1%            0.0000
5%            0.0000
10%           0.0030
25%           0.0299
50%           0.1542
75%           0.5590
90%           0.9813
95%           1.0000
99%           1.0930
99.5%         1.3663
max      50,708.0000
Name: revolving_utilization, dtype: float64

Распределение значений среди объектов с экстремальным `revolving_utilization`.

In [55]:
make_num_charactericstic_table(
    data.query(f"revolving_utilization > {overall_characteristics.loc["revolving_utilization", "99.5%"]}")
)

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,750.0000,"1,146.5882","3,344.0969",1.3682,1.3704,1.3731,1.3970,1.4328,1.5601,1.9931,769.5000,"3,595.4000","6,057.2500","13,718.3200","20,892.9300","50,708.0000"
age,750.0000,45.8440,13.3227,21.0000,22.0000,23.0000,27.0000,29.0000,35.0000,45.0000,55.0000,64.0000,69.5500,81.0000,83.5100,87.0000
num_30_59_days_late,750.0000,0.6747,1.0632,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,4.0000,5.0000,7.0000
debt_ratio,750.0000,322.2623,"1,153.4965",0.0008,0.0018,0.0032,0.0176,0.0504,0.1600,0.3606,2.6432,974.1000,"2,295.3500","4,538.3300","5,251.5350","21,395.0000"
monthly_income,571.0000,"6,351.6130","6,459.4001",0.0000,0.0000,28.3000,"1,259.5000","1,800.0000","2,929.5000","4,600.0000","7,651.5000","12,007.0000","16,250.0000","36,751.2000","44,700.6000","69,520.0000"
num_open_credit_lines,750.0000,5.5693,3.8472,0.0000,0.0000,0.4900,1.0000,2.0000,3.0000,5.0000,7.0000,10.0000,13.0000,19.0200,21.2550,24.0000
num_90_days_late,750.0000,0.8413,1.6935,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,4.0000,7.5100,8.2550,15.0000
num_real_estate_loans,750.0000,0.7093,0.9795,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,3.5100,5.0000,9.0000
num_60_89_days_late,750.0000,0.4573,0.9360,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,4.0000,5.2550,7.0000
num_dependents,727.0000,0.8294,1.1552,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,3.0000,3.0000,4.0000,4.3700,5.0000


Высокие значения, скорее всего, либо являются ошибочными, либо имеют уже другой смысл. Скорее всего, экстремальные значения (скажем больше 3), являются уже просто абсолютным значением долга. То есть значение 769.5 значит, что заёмщик должен 769.5 долларов или евро.

**Наличие положительных значений в признаке при отсутствующих открытых кредитных линиях.**

Наличие положительных значений в признаке при отсутствующих открытых кредитных линиях.

In [56]:
data.query("(revolving_utilization > 0) & (num_open_credit_lines == 0)").shape[0]

1888

Представители.

In [57]:
data.query("(revolving_utilization > 0) & (num_open_credit_lines == 0)").sample(10)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
40088,0,1.0000,56,0,0.0000,NaN,0,0,0,0,0.0000
14587,0,1.0000,37,0,0.0000,"1,500.0000",0,0,0,0,0.0000
13058,0,1.0000,38,0,0.0564,"2,674.0000",0,0,0,0,1.0000
105401,0,1.0000,30,0,0.0000,"2,542.0000",0,0,0,0,2.0000
60204,1,1.0000,26,0,0.0000,NaN,0,1,0,0,0.0000
35033,0,1.0000,25,0,0.0088,"2,494.0000",0,1,0,0,0.0000
73148,0,1.0000,51,0,0.0000,"6,928.0000",0,0,0,0,1.0000
76383,0,1.0000,26,0,0.0000,"3,500.0000",0,0,0,0,2.0000
120980,0,1.0000,22,0,20.0000,NaN,0,0,0,0,0.0000
3934,0,1.0000,33,0,0.0000,400.0000,0,0,0,0,1.0000


Количество с `revolving_utilization` != 1.0.

In [58]:
data.query("(abs(revolving_utilization - 1.0) > 10e-4) & (num_open_credit_lines == 0)").shape[0]

10

Количество людей с `revolving_utilization` == 1.0.

In [59]:
data.query("abs(revolving_utilization - 1.0) < 10e-4").shape[0]

10445

Количество людей с `revolving_utilization` == 1.0 и `debt_ratio` > 0.

In [60]:
data.query("(abs(revolving_utilization - 1.0) < 10e-4) & (debt_ratio > 0)").shape[0]

8595

Исходя из определения признаков нельзя объяснить полученные значения. Можно было предположить, что `revolving_utilization` = 1.0 сигнализирует об отсутствии кредитных лимитов или долгов. Однако такие сценарии подтвердить вычислениями достоверно нельзя.

**Вывод по признаку:**

Я бы сказал, что в признаке присутствует одна ключевая проблема - экстремальные значения, природа которых, скорее всего, заключается в смене смысла признака (отношение сменяется на абсолютное значение долга). Проблему с положительным `revolving_utilization` при отсутствии кредитных линий, можно списать на незначительную (таких представителей не много, меньше процента, плюс проследить до конца природу такого являения, чтобы признать его ошибкой, не представляется возможным в рамках текущей документации).

In [61]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "revolving_utilization",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не должен принимать значения сильно выше 1;\n"
    "3. положителельные значения признака должны сопровождаться положительными значениями `num_open_credit_lines`.",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. экстремально большие значения, вероятнее всего, объясняются сменой смысла признака: из отношения к абсолютным значениям;\n"
    "3. в рамках текущей документации сделать вывод о ситуации, когда `revolving_utilization` положителен, "
    "а `num_open_credit_lines` = 0, не представляется возможным",
    "1. сохранить исходное значение; отдельно отметить экстремальные значения; способ обработки определить после анализа их связи с риском."
]

## Анализ качества признака `age`

**age** - возраст заёмщика в годах.

Исходя из определения признака:
1. Он не должен принимать отрицательные значения;
2. Значения должны быть не ниже 18 лет (возраст совершеннолетия);
3. Значения не должны быть сильно большими и превышать какие-то разумные пределы, скажем 90 лет.

**Проверим это:**

**Наличие отрицательных значений.**

Количество заёмщиков с отрицательным значением возраста.

In [62]:
data.query("age < 0").shape[0]

0

Отрицательные значения отсутствуют.

**Распределение значений признака.**

In [63]:
overall_characteristics.loc["age"]

count   150,000.0000
mean         52.2952
std          14.7719
min           0.0000
0.5%         23.0000
1%           24.0000
5%           29.0000
10%          33.0000
25%          41.0000
50%          52.0000
75%          63.0000
90%          72.0000
95%          78.0000
99%          87.0000
99.5%        89.0000
max         109.0000
Name: age, dtype: float64

Есть как представитель с возрастом 0 лет, так и представители с возрастом больше 90 лет.

**Наличие низких значений.**

Количество людей с возрастом менее 18 лет.

In [64]:
data.query("age < 18").shape[0]

1

Представитель с возрастом менее 18 лет.

In [65]:
data.query("age < 18")

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
65695,0,1.0000,0,1,0.4369,"6,000.0000",6,0,2,0,2.0000


Это явно ошибка. Стоит считать, что возраст просто пропущен.

**Наличие высоких значений**

Количество представителей с значением возраста больше 90 лет.

In [66]:
data.query("age > 90").shape[0]

489

Представители с возрастом свыше 90 лет.

In [67]:
data.query("age > 90").sample(10)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
21629,0,0.0194,92,0,0.0028,"9,000.0000",5,0,0,0,0.0000
53654,0,0.0079,92,0,0.0011,"8,333.0000",1,0,0,0,0.0000
92718,1,0.1621,99,1,150.0000,NaN,2,0,0,0,NaN
132842,1,0.5789,91,0,0.2976,"4,166.0000",4,4,0,1,1.0000
49575,0,0.0371,92,0,313.0000,NaN,7,0,0,1,0.0000
33379,0,0.0441,95,0,695.0000,NaN,3,0,1,0,NaN
133885,0,0.0000,91,0,0.0000,NaN,1,0,0,0,0.0000
39846,1,0.0013,99,0,0.0005,"6,500.0000",15,0,0,0,0.0000
25545,0,0.0117,92,0,0.0023,"3,500.0000",3,0,0,0,0.0000
23235,0,0.0013,92,0,0.0003,"3,500.0000",5,0,0,0,0.0000


Распределение значений признаков среди людей с возрастом свыше 100 лет.

In [68]:
make_num_charactericstic_table(data.query("age > 90"))

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,489.0000,0.1328,0.2908,0.0000,0.0000,0.0000,0.0000,0.0000,0.0025,0.0139,0.0590,0.4991,1.0000,1.0000,1.0000,1.0000
age,489.0000,93.1656,2.6464,91.0000,91.0000,91.0000,91.0000,91.0000,91.0000,92.0000,94.0000,96.0000,98.0000,103.0000,106.1200,109.0000
num_30_59_days_late,489.0000,0.1207,0.4702,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,5.0000
debt_ratio,489.0000,156.2554,"1,052.8753",0.0000,0.0000,0.0000,0.0000,0.0000,0.0007,0.0496,8.0000,110.8000,572.6000,"2,867.8000","3,719.2000","20,809.0000"
monthly_income,255.0000,"5,897.8549","13,139.6720",0.0000,0.0000,0.0000,"1,100.7000","1,665.4000","2,658.0000","4,079.0000","6,450.0000","9,248.4000","10,500.0000","20,383.1800","40,104.2700","203,500.0000"
num_open_credit_lines,489.0000,5.4908,4.0590,0.0000,0.0000,1.0000,1.0000,1.0000,3.0000,5.0000,7.0000,11.0000,13.0000,18.0000,26.1200,30.0000
num_90_days_late,489.0000,0.0286,0.2378,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,4.0000
num_real_estate_loans,489.0000,0.2045,0.5348,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,2.0000,3.1200,4.0000
num_60_89_days_late,489.0000,0.0327,0.2193,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,3.0000
num_dependents,399.0000,0.0752,0.2640,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000


В целом нельзя сказать что-то определённое о таких заёмщиках. С одной стороны хочется сказать, что просто банк не закрыл с ними сотрудничество и продляет с ними договор, с другой стороны у некоторых заёмщиков есть открытые кредитные линии и даже просрочки, поэтому трудно дать какой-то вердикт по таким представителям. Возраст, конечно, сомнительный, но при этом нет законов, которые запрещали бы пожилым людям иметь кредиты и кредитные линии.

**Вывод по признаку:**

Значения признака в целом выглядят нормальными и адекватными, за исключением заёмщика с 0 лет - это явно ошибка (такого представителя можно либо удалить, либо, что разумнее, добавить флаг об ошибке в возрасте).

In [69]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "age",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не может принимать значения меньше 18 лет (возраст совершеннолетия);\n"
    "3. признак не должен принимать значения сильно выше 90.",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. есть один представитель с возрастом 0 лет - это явно ошибка;\n"
    "3. сделать вывод об ошибочности представителей с экстремальными значениями в рамках текущей "
    "документации невозможно, считаем их нормальными.",
    "1. сделать флаг об ошибке в признаке (возраст ниже 18 лет)."
]

## Анализ качества признаков `num_30_59_days_late`, `num_60_89_days_late` и `num_90_days_late`

**num_30_59_days_late, num_60_89_days_late, num_90_days_late** - наличие просрочек у клиента в 30-59, 60-89, 90+ дней соответственно за последние 2 года (да, да, нет).

Исходя из смысла признаков:
1. Признаки не могут принимать отрицательные значения;
2. Признаки не могут быть огромными: скорее всего для большинства людей может быть одобрено 1-2 кредита, для инвесторов и бизнесменов может быть одобрено много, но всё-таки не 50 и не 100 кредитов, по которым могут быть просрочки. Другими словами, значения если и могут быть большими, то только умеренно.

**Проверим это:**

**Наличие отрицательных значений.**

In [70]:
data.query("(num_30_59_days_late < 0) | (num_60_89_days_late < 0) | (num_90_days_late < 0)").shape[0]

0

Отрицательные значения отсутствуют.

**Наличие экстремальных значений.**

Посмотрим на распределение значений у этих признаков.

In [71]:
overall_characteristics.loc[["num_30_59_days_late", "num_60_89_days_late", "num_90_days_late"]]

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
num_30_59_days_late,"150,000.0000",0.4210,4.1928,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,5.0000,98.0000
num_60_89_days_late,"150,000.0000",0.2404,4.1552,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,98.0000
num_90_days_late,"150,000.0000",0.2660,4.1693,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,4.0000,98.0000


Посмотрим на представителей экстремальных значений.

In [72]:
data.query("(num_30_59_days_late > 10) | (num_60_89_days_late > 10) | (num_90_days_late > 10)").sample(10)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
124987,1,1.0000,42,98,0.0000,"3,830.0000",0,98,0,98,0.0000
55072,0,1.0000,56,98,0.0000,"7,781.0000",0,98,0,98,5.0000
19332,0,1.0000,24,98,0.0000,NaN,0,98,0,98,0.0000
76698,0,1.0000,23,98,0.0000,NaN,0,98,0,98,NaN
23447,1,1.0000,35,98,0.0000,"2,250.0000",0,98,0,98,2.0000
22463,0,1.0000,26,98,0.0000,NaN,0,98,0,98,0.0000
84167,1,1.0000,29,96,0.0000,"2,800.0000",0,96,0,96,2.0000
47135,1,1.0000,24,98,0.0351,"2,650.0000",0,98,0,98,1.0000
136545,1,1.0000,54,98,0.0594,"1,800.0000",0,98,0,98,0.0000
23287,1,1.0000,25,98,0.0000,NaN,0,98,0,98,0.0000


Посмотрим уникальные значения признаков.

In [73]:
[
    sorted(data["num_30_59_days_late"].unique().tolist(), reverse=True),
    sorted(data["num_60_89_days_late"].unique().tolist(), reverse=True),
    sorted(data["num_90_days_late"].unique().tolist(), reverse=True)
]

[[98, 96, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
 [98, 96, 11, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
 [98, 96, 17, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]]

Явно выделяются значения 98 и 96, остальные значения не выглядят невозможными. Кроме того, можно заметить, что 98 и 96 стоит сразу во всех 3 признаках одновременно, что наталкивает на мысль, что данные значения являются какими-либо специальными кодами.

Проверим, данную гипотезу.

In [74]:
(
    data.query(
        "(num_30_59_days_late == num_60_89_days_late == num_90_days_late == 96) |"
        "(num_30_59_days_late == num_60_89_days_late == num_90_days_late == 98)"
    ).shape[0],
    data.query(
        "(num_30_59_days_late == 96) | (num_30_59_days_late == 98) |"
        "(num_60_89_days_late == 96) | (num_60_89_days_late == 98) |"
        "(num_90_days_late == 96) | (num_90_days_late == 98)"
    ).shape[0]
)

(269, 269)

Гипотеза подтвердилась - данные значения, если и встречаются, то сразу во всех трёх признаках одинаковые. Это говорит о том, что такие значения являются незадокументированными системными кодами. Такие значения можно просто в дальнейшем подсвечивать флагом.

**Вывод по признакам:**

В признаках нет аномалий, кроме значений 96 и 98, которые, скорее всего, являются незадокументированными системными кодами, которые требуют выделения при помощи соответствующих флагов.

In [75]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "num_30_59_days_late / num_60_89_days_late / num_90_days_late",
    "1. признаки не могут принимать отрицательных значений;\n"
    "2. признаки не могут принимать слишком большие значения (порядка 50, сотен и так далее).",
    "1. признаки не содержат ошибок, связанных с отрицательными значениями;\n"
    "2. признаки не содержат ненормально больших значений, кроме 96 и 98;\n"
    "3. значения 96 и 98 являются системными кодами. ",
    "1. сделать флаги о системных кодах 96 и 98."
]

## Анализ качества признака `debt_ratio`

**debt_ratio** - ежемесячные выплаты по долгам, алименты и расходы на проживание, делённые на ежемесячный валовый доход.

Исходя из определения признака:
1. Не может быть отрицательных значений;
2. По величине значений, к сожалению, ничего сказать нельзя, так как:
    1. у заёмщика может быть высокая кварплата и при этом отсутствовать доход (например, человек живёт за счёт нетрудовых доходов (или неофициальных));
    2. заёмщик может потерять доход частично или полностью из-за чего соотношение может стать неадекватно большим. То есть никак не оговорено, как производятся расчёты в случае отсутствия знаменателя.

**Проверим это:**

**Наличие отрицательных значений.**

In [76]:
data.query("debt_ratio < 0").shape[0]

0

Отрицательные значения отсутствуют.

**Аномальные значения.**

Распределение значений признака.

In [77]:
overall_characteristics.loc["debt_ratio"]

count   150,000.0000
mean        353.0051
std       2,037.8185
min           0.0000
0.5%          0.0000
1%            0.0000
5%            0.0043
10%           0.0309
25%           0.1751
50%           0.3665
75%           0.8683
90%       1,267.0000
95%       2,449.0000
99%       4,979.0400
99.5%     6,186.0100
max     329,664.0000
Name: debt_ratio, dtype: float64

Распределение характеристик среди эксремальных значений признака.

In [78]:
make_num_charactericstic_table(
    data.query("debt_ratio > 1.0")
)

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"35,137.0000",7.0762,239.5104,0.0000,0.0000,0.0000,0.0000,0.0022,0.0225,0.1053,0.4904,0.9750,1.0000,1.1174,1.4810,"22,198.0000"
age,"35,137.0000",55.0063,14.9089,21.0000,23.0000,24.0000,30.0000,35.0000,44.0000,55.0000,65.0000,75.0000,80.0000,88.0000,90.0000,109.0000
num_30_59_days_late,"35,137.0000",0.2993,2.8416,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,3.0000,5.0000,98.0000
debt_ratio,"35,137.0000","1,505.9896","3,999.0268",1.0005,1.0154,1.0305,1.2085,1.6936,42.0000,907.0000,"2,210.0000","3,586.4000","4,716.0000","7,927.6400","10,047.9600","329,664.0000"
monthly_income,"7,233.0000","2,258.9551","2,831.1682",0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,"1,577.0000","3,333.0000","5,416.0000","7,103.2000","12,325.9600","14,500.0000","70,000.0000"
num_open_credit_lines,"35,137.0000",8.0717,5.0984,0.0000,0.0000,1.0000,2.0000,3.0000,4.0000,7.0000,11.0000,15.0000,18.0000,24.0000,27.0000,58.0000
num_90_days_late,"35,137.0000",0.1660,2.8062,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,98.0000
num_real_estate_loans,"35,137.0000",1.0829,1.2777,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,3.0000,5.0000,7.0000,54.0000
num_60_89_days_late,"35,137.0000",0.1404,2.7843,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,2.0000,98.0000
num_dependents,"31,689.0000",0.4366,0.9279,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,3.0000,4.0000,4.0000,10.0000


In [79]:
make_num_charactericstic_table(
    data.query("debt_ratio > 200.0")
)

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"23,088.0000",7.5227,242.2625,0.0000,0.0000,0.0000,0.0000,0.0012,0.0246,0.1165,0.4697,0.9626,1.0000,1.1042,1.4315,"22,198.0000"
age,"23,088.0000",54.0864,13.4228,21.0000,25.0000,26.0000,31.0000,36.0000,44.0000,55.0000,63.0000,71.0000,76.0000,84.0000,87.0000,109.0000
num_30_59_days_late,"23,088.0000",0.2333,1.1278,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,3.0000,4.0000,98.0000
debt_ratio,"23,088.0000","2,274.8297","4,755.3639",201.0000,215.0000,227.8700,342.0000,477.7000,940.7500,"1,755.0000","2,853.0000","4,249.0000","5,412.9500","9,151.2600","11,871.7750","329,664.0000"
monthly_income,"1,570.0000",0.2866,0.5796,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,10.0000
num_open_credit_lines,"23,088.0000",8.4081,4.7878,0.0000,1.0000,1.0000,2.0000,3.0000,5.0000,8.0000,11.0000,15.0000,18.0000,24.0000,26.0000,45.0000
num_90_days_late,"23,088.0000",0.0956,1.0327,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,3.0000,98.0000
num_real_estate_loans,"23,088.0000",1.1897,1.0339,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,2.0000,2.0000,3.0000,4.0000,5.5650,23.0000
num_60_89_days_late,"23,088.0000",0.0732,0.9721,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,2.0000,98.0000
num_dependents,"20,958.0000",0.4315,0.9309,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,3.0000,4.0000,4.0000,10.0000


Можно заметить, что крайне экстремальные значения признака, как правило, сопровождаются отсутствующим `monthly_income`, то есть отсутствующим знаменателем. Кроме того, сами значения, скорее всего, как в и `revolving_utilization` имеют уже не относительное, а абсолютное выражение.

**Вывод по признаку:**
1. Ошибочных значений в признаке нет;
2. Экстремальные значения предположительно бывают двух типов:
    1. частичная потеря зароботка или просто маленький `monthly_income`, которые делают очень большим итоговое отношение `debt_ratio`;
    2. отсутствующий `monthly_income` переводит значения из относительных в абсолюные (просто сумма выплат в месяц).

In [80]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "debt_ratio",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не должен принимать значения сильно выше 1.",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. экстремальные значения имеют распределение, несовместимое с буквальной интерпретацией "
    "показателя как обычного ratio. Возможными объяснениями являются особенности расчёта показателя "
    "при малом/нулевом знаменателе, изменение семантики поля или ошибки данных. Исходная "
    "документация не позволяет установить точную причину.",
    "1. сохранить исходное значение; отдельно отметить экстремальные значения; способ обработки определить после анализа их связи с риском."
]

## Анализ качества признака `monthly_income`

**monthly_income** - ежемесячный валовый доход заёмщика.

Исходя из определения признака:
1. Признак не может принимать отрицательных значений;
2. Признак не должен содержать очень много больших или малых значений.

**Проверим это:**

**Наличие отрицательных значений.**

In [81]:
data.query("monthly_income < 0").shape[0]

0

Отрицательные значения отсутствуют.

**Наличие экстремальных значений.**

Распределение значений признака.

In [82]:
overall_characteristics.loc["monthly_income"]

count     120,269.0000
mean        6,670.2212
std        14,384.6742
min             0.0000
0.5%            0.0000
1%              0.0000
5%          1,300.0000
10%         2,005.0000
25%         3,400.0000
50%         5,400.0000
75%         8,249.0000
90%        11,666.0000
95%        14,587.6000
99%        25,000.0000
99.5%      35,000.0000
max     3,008,750.0000
Name: monthly_income, dtype: float64

В целом нет каких-либо аномальных значений. 3.000.000 пусть и много, но возможно. Таким образом, проблем не видно.

**Вывод по признаку:**

В признаке нет каких-либо аномальных или ошибочных значений (мешают только пропуски).

In [83]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "monthly_income",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не может принимать совсем больших значений, порядка десятков и сотен миллионов.",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. признак не содержит ошибок, связанных с экстремальными значениями.",
    "Ничего не предпринимать."
]

## Анализ качества признака `num_open_credit_lines`

**num_open_credit_lines** - количесво открытых кредитов, например автокредит или ипотека, и кредитных линий, например кредитных карт.

Исходя из определения признака:
1. Признак не может принимать отрицательные значения;
2. Признак не может иметь слишком больших значений, порядка 100, так как трудно представить человека, имеющего столько кредитов.

**Наличие отрицательных значений.**

In [84]:
data.query("num_open_credit_lines < 0").shape[0]

0

Отрицательные значения отсутствуют.

**Наличие экстремальных значений.**

Распределение значений признака.

In [85]:
overall_characteristics.loc["num_open_credit_lines"]

count   150,000.0000
mean          8.4528
std           5.1460
min           0.0000
0.5%          0.0000
1%            0.0000
5%            2.0000
10%           3.0000
25%           5.0000
50%           8.0000
75%          11.0000
90%          15.0000
95%          18.0000
99%          24.0000
99.5%        27.0000
max          58.0000
Name: num_open_credit_lines, dtype: float64

Экстремальные значения есть, но их нельзя признать ошибочными, так как экстремальные значения редки и в пределах возможных значений.

**Вывод по признаку:**
1. Ошибочных значений в признаке не наблюдается;
2. Экстремальные значения находятся в пределах нормы.

In [86]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "num_open_credit_lines",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не может принимать сильно больших значений (порядка сотен).",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. признак не содержит ошибок, связанных с экстремальными значениями.",
    "Ничего не предпринимать."
]

## Анализ качества признака `num_real_estate_loans`

**num_real_estate_loans** - количество ипотечных и других кредитов, связанных с недвижимостью, включая кредитные линии под залог жилья.

Исходя из определения признака:
1. Признак не может принимать отрицательные значения;
2. Признак не может принимать огромные значения, только в разумных пределах у каких-либо инвесторов;
3. `num_real_estate_loans` не может превышать значение `num_open_credit_lines`.

**Проверим это:**

**Наличие отрицательных значений.**

In [87]:
data.query("num_real_estate_loans < 0").shape[0]

0

Отрицательные значения отсутствуют.

**Наличие экстремальных значений.**

Распределение значений признака.

In [88]:
overall_characteristics.loc["num_real_estate_loans"]

count   150,000.0000
mean          1.0182
std           1.1298
min           0.0000
0.5%          0.0000
1%            0.0000
5%            0.0000
10%           0.0000
25%           0.0000
50%           1.0000
75%           2.0000
90%           2.0000
95%           3.0000
99%           4.0000
99.5%         6.0000
max          54.0000
Name: num_real_estate_loans, dtype: float64

Представители с экстремальными значениями.

In [89]:
data.query(
    f"num_real_estate_loans > {overall_characteristics.loc["num_real_estate_loans", "99.5%"]}"
).sample(15)

,target,revolving_utilization,age,num_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,num_90_days_late,num_real_estate_loans,num_60_89_days_late,num_dependents
133512,0,0.0264,56,0,0.5469,"7,000.0000",16,0,7,0,0.0000
68909,0,0.2977,44,0,1.2375,"12,400.0000",42,0,25,0,2.0000
48689,0,0.4948,43,0,1.9026,"7,143.0000",12,0,7,0,2.0000
59005,0,0.3994,64,0,"10,850.0000",NaN,18,0,11,0,0.0000
19844,0,0.4004,59,0,"8,741.0000",NaN,17,0,7,0,0.0000
27372,0,0.4296,57,0,"9,211.0000",NaN,11,0,7,0,0.0000
98015,0,0.2625,43,0,0.3400,"23,500.0000",25,0,7,0,0.0000
138070,0,0.0062,63,0,"14,336.0000",NaN,19,0,9,0,0.0000
145712,0,0.0167,49,0,1.4135,"5,800.0000",13,0,8,0,4.0000
62473,0,0.1680,63,0,"9,541.0000",NaN,13,0,7,0,0.0000


Пусть есть большие значения, они всё же не выглядят ошибочными или невозможными. Скорее всего, тут ошибок нет.

**Превышение значений `num_real_estate_loans` над значениями `num_open_credit_lines`.**

In [90]:
data.query("num_open_credit_lines < num_real_estate_loans").shape[0]

0

**Вывод по признаку:**

В признаке ошибок нет.

In [91]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "num_real_estate_loans",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не может принимать сильно больших значений (порядка сотен).",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. признак не содержит ошибок, связанных с экстремальными значениями.",
    "Ничего не предпринимать."
]

## Анализ качества признака `num_dependents`

**num_dependents** - количество иждевенцев в семье, не включая самого заёмщика: супруг/супруга, дети и так далее.

Исходя из определения признака:
1. Признак не может принимать отрицательные значения;
2. Признак не может принимать слишком большие значения, порядка нескольких десятков.

**Проверим это:**

**Наличие отрицательных значений.**

In [92]:
data.query("num_dependents < 0").shape[0]

0

Отрицательные значения отсутствуют.

**Наличие экстремальных значений.**

Распределение значений признака.

In [93]:
overall_characteristics.loc["num_dependents"]

count   146,076.0000
mean          0.7572
std           1.1151
min           0.0000
0.5%          0.0000
1%            0.0000
5%            0.0000
10%           0.0000
25%           0.0000
50%           0.0000
75%           1.0000
90%           2.0000
95%           3.0000
99%           4.0000
99.5%         5.0000
max          20.0000
Name: num_dependents, dtype: float64

Подозрительных значений нет.

**Вывод по признаку:**

Проблем в признаке нет.

In [94]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "num_dependents",
    "1. признак не может принимать отрицательных значений;\n"
    "2. признак не может принимать сильно больших значений (порядка сотен).",
    "1. признак не содержит ошибок, связанных с отрицательными значениями;\n"
    "2. признак не содержит ошибок, связанных с экстремальными значениями.",
    "Ничего не предпринимать."
]

## Анализ пропусков

В рамках бизнес задач нужно проанализировать связь пропусков с targetом.

### Анализ пропусков `monthly_income`

**Распределения:**

Распределение значений признаков среди объектов с пропущенным `monthly_income`.

In [95]:
nan_mon_inc_characteristics = make_num_charactericstic_table(
    data[data["monthly_income"].isna()]
)

In [96]:
navigation["количественные характеристики признаков у объектов, с пропущенным monthly_income"] = "nan_mon_inc_characteristics"

In [97]:
nan_mon_inc_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"29,731.0000",6.6494,217.8149,0.0000,0.0000,0.0000,0.0000,0.0000,0.0160,0.0817,0.4405,1.0000,1.0000,1.1136,1.4850,"22,198.0000"
age,"29,731.0000",56.3623,15.4388,21.0000,22.0000,24.0000,30.0000,35.0000,46.0000,57.0000,67.0000,77.0000,82.0000,90.0000,92.0000,109.0000
num_30_59_days_late,"29,731.0000",0.5799,6.2554,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,4.0000,5.0000,98.0000
debt_ratio,"29,731.0000","1,673.3966","4,248.3729",0.0000,0.0000,0.0000,0.0000,11.0000,123.0000,"1,159.0000","2,382.0000","3,785.0000","4,902.5000","8,084.5000","10,220.1000","329,664.0000"
monthly_income,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_open_credit_lines,"29,731.0000",7.2161,4.8427,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,6.0000,10.0000,14.0000,16.0000,23.0000,25.0000,45.0000
num_90_days_late,"29,731.0000",0.4846,6.2504,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,6.0000,98.0000
num_real_estate_loans,"29,731.0000",0.8715,1.0343,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,2.0000,2.0000,4.0000,5.0000,23.0000
num_60_89_days_late,"29,731.0000",0.4530,6.2421,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,3.3500,98.0000
num_dependents,"25,807.0000",0.3163,0.8099,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,4.0000,9.0000


Распределение значений признаков в рамках всего датасета.

In [98]:
overall_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"150,000.0000",6.0484,249.7554,0.0000,0.0000,0.0000,0.0000,0.0030,0.0299,0.1542,0.5590,0.9813,1.0000,1.0930,1.3663,"50,708.0000"
age,"150,000.0000",52.2952,14.7719,0.0000,23.0000,24.0000,29.0000,33.0000,41.0000,52.0000,63.0000,72.0000,78.0000,87.0000,89.0000,109.0000
num_30_59_days_late,"150,000.0000",0.4210,4.1928,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,5.0000,98.0000
debt_ratio,"150,000.0000",353.0051,"2,037.8185",0.0000,0.0000,0.0000,0.0043,0.0309,0.1751,0.3665,0.8683,"1,267.0000","2,449.0000","4,979.0400","6,186.0100","329,664.0000"
monthly_income,"120,269.0000","6,670.2212","14,384.6742",0.0000,0.0000,0.0000,"1,300.0000","2,005.0000","3,400.0000","5,400.0000","8,249.0000","11,666.0000","14,587.6000","25,000.0000","35,000.0000","3,008,750.0000"
num_open_credit_lines,"150,000.0000",8.4528,5.1460,0.0000,0.0000,0.0000,2.0000,3.0000,5.0000,8.0000,11.0000,15.0000,18.0000,24.0000,27.0000,58.0000
num_90_days_late,"150,000.0000",0.2660,4.1693,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,4.0000,98.0000
num_real_estate_loans,"150,000.0000",1.0182,1.1298,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,3.0000,4.0000,6.0000,54.0000
num_60_89_days_late,"150,000.0000",0.2404,4.1552,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,98.0000
num_dependents,"146,076.0000",0.7572,1.1151,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,4.0000,5.0000,20.0000


В целом распределения не отличаются значительно от распределений по датасету в целом. Нельзя сказать, что представители, у которых нет данных о ежемесячном доходе особенные.

**bad rate:**

bad rate объектов с пропусками.

In [99]:
bad_rate_nan_monthly_income = round(data[data["monthly_income"].isna()]["target"].mean(), 4)

In [100]:
navigation["bad rate объектов с пустым monthly_income"] = "bad_rate_nan_monthly_income"

Доверительный интервал bad rate объектов с пропусками в monthly_income.

In [101]:
temp = data[data["monthly_income"].isna()]

ci_low_nan_mon_inc_bad_rate, ci_high_nan_mon_inc_bad_rate = proportion_confint(
    count=temp["target"].sum(),
    nobs=temp.shape[0],
    alpha=0.05,
    method="wilson"
)

ci_nan_mon_inc_bad_rate = (round(ci_low_nan_mon_inc_bad_rate, 4), round(ci_high_nan_mon_inc_bad_rate, 4))

In [102]:
navigation["доверительный интервал bad rate объектов с пропусками monthly_income"] = "ci_nan_mon_inc_bad_rate"

Сравнение ситуации по датасету в целом.

In [103]:
overall_bad_rate, ci_overall_bad_rate

(np.float64(0.0668), (0.0656, 0.0681))

In [104]:
bad_rate_nan_monthly_income, ci_nan_mon_inc_bad_rate

(np.float64(0.0561), (0.0536, 0.0588))

Проверим значимости отличия.

In [105]:
mask = data["monthly_income"].isna()

make_z_test(data=data, mask=mask, alpha=0.05)

np.True_

При уровне значимости 0.05 отличие статистически значимо, но в абсолютных значениях отличие не велико.

**Вывод по пропускам в monthly_income:**
1. Распределения признаков (кроме `debt_ratio` не имеют значительных отличий от датасета в целом);
2. bad rate отличается от датасета в целом, но не настолько значительно, чтобы делать по этому однозначные выводы;
3. Стоит сделать флаг, который будет сигнализировать об отсутствии данных о доходе заёмщика.

In [106]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "missing monthly_income",
    "Нет никаких ограничений.",
    "Объекты с пропусками не сильно отличаются от датасета в целом.",
    "1. Сделать флаг об отсутствии данных о ежемесячном доходе."
]

### Анализ пропусков `num_dependents`

**Распределения:**

Распределение значений признаков среди объектов с пропущенным `num_dependents`.

In [107]:
nan_num_deps_characteristics = make_num_charactericstic_table(
    data[data["num_dependents"].isna()]
)

In [108]:
navigation["количественные характеристики признаков у объектов с пропущенным num_dependents"] = "nan_num_deps_characteristics"

In [109]:
nan_num_deps_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"3,924.0000",10.7451,237.6992,0.0000,0.0000,0.0000,0.0000,0.0000,0.0085,0.0475,0.2682,1.0000,1.0000,1.1026,1.5517,"10,821.0000"
age,"3,924.0000",59.5889,18.6342,21.0000,21.0000,22.0000,25.0000,30.0000,48.0000,61.0000,74.0000,83.0000,88.0000,93.0000,95.0000,109.0000
num_30_59_days_late,"3,924.0000",0.9083,8.6794,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,4.0000,98.0000,98.0000
debt_ratio,"3,924.0000","1,083.8122","4,186.7318",0.0000,0.0000,0.0000,0.0000,0.0000,21.0000,358.0000,"1,559.0000","2,830.0000","3,688.1000","6,042.5500","7,223.6800","220,516.0000"
monthly_income,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_open_credit_lines,"3,924.0000",5.6042,4.0964,0.0000,0.0000,0.0000,1.0000,1.0000,3.0000,5.0000,8.0000,11.0000,14.0000,19.0000,21.0000,30.0000
num_90_days_late,"3,924.0000",0.8346,8.6792,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,3.0000,98.0000,98.0000
num_real_estate_loans,"3,924.0000",0.5910,0.9145,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,4.0000,4.0000,15.0000
num_60_89_days_late,"3,924.0000",0.8122,8.6780,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.7700,98.0000,98.0000
num_dependents,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Распределение значений признаков среди объектов с `num_dependents` == 0.

In [110]:
zero_num_deps_characteristics = make_num_charactericstic_table(
    data[data["num_dependents"] == 0]
)

In [111]:
navigation["количественные характеристики признаков у объектов с нулевым num_dependents"] = "zero_num_deps_characteristics"

In [112]:
zero_num_deps_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"86,902.0000",5.6664,263.0430,0.0000,0.0000,0.0000,0.0000,0.0018,0.0231,0.1188,0.5051,0.9792,1.0000,1.0772,1.3273,"50,708.0000"
age,"86,902.0000",54.3966,16.0918,21.0000,22.0000,23.0000,27.0000,31.0000,42.0000,56.0000,66.0000,75.0000,80.0000,88.0000,90.0000,107.0000
num_30_59_days_late,"86,902.0000",0.4232,4.5876,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,3.0000,5.0000,98.0000
debt_ratio,"86,902.0000",418.9466,"2,091.6543",0.0000,0.0000,0.0000,0.0020,0.0130,0.1496,0.3724,2.9111,"1,589.0000","2,696.9500","5,237.9700","6,600.4450","329,664.0000"
monthly_income,"65,456.0000","5,873.4115","10,545.8570",0.0000,0.0000,0.0000,"1,000.0000","1,700.0000","3,000.0000","4,783.0000","7,400.0000","10,500.0000","12,972.0000","21,909.0000","30,000.0000","1,794,060.0000"
num_open_credit_lines,"86,902.0000",8.1581,5.1841,0.0000,0.0000,0.0000,2.0000,2.0000,4.0000,7.0000,11.0000,15.0000,18.0000,24.0000,27.4950,58.0000
num_90_days_late,"86,902.0000",0.2919,4.5710,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,98.0000
num_real_estate_loans,"86,902.0000",0.9056,1.1123,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,2.0000,3.0000,4.0000,5.0000,54.0000
num_60_89_days_late,"86,902.0000",0.2680,4.5593,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,2.0000,98.0000
num_dependents,"86,902.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


Распределение значений признаков в рамках всего датасета.

In [113]:
overall_characteristics

,count,mean,std,min,0.5%,1%,5%,10%,25%,50%,75%,90%,95%,99%,99.5%,max
revolving_utilization,"150,000.0000",6.0484,249.7554,0.0000,0.0000,0.0000,0.0000,0.0030,0.0299,0.1542,0.5590,0.9813,1.0000,1.0930,1.3663,"50,708.0000"
age,"150,000.0000",52.2952,14.7719,0.0000,23.0000,24.0000,29.0000,33.0000,41.0000,52.0000,63.0000,72.0000,78.0000,87.0000,89.0000,109.0000
num_30_59_days_late,"150,000.0000",0.4210,4.1928,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,4.0000,5.0000,98.0000
debt_ratio,"150,000.0000",353.0051,"2,037.8185",0.0000,0.0000,0.0000,0.0043,0.0309,0.1751,0.3665,0.8683,"1,267.0000","2,449.0000","4,979.0400","6,186.0100","329,664.0000"
monthly_income,"120,269.0000","6,670.2212","14,384.6742",0.0000,0.0000,0.0000,"1,300.0000","2,005.0000","3,400.0000","5,400.0000","8,249.0000","11,666.0000","14,587.6000","25,000.0000","35,000.0000","3,008,750.0000"
num_open_credit_lines,"150,000.0000",8.4528,5.1460,0.0000,0.0000,0.0000,2.0000,3.0000,5.0000,8.0000,11.0000,15.0000,18.0000,24.0000,27.0000,58.0000
num_90_days_late,"150,000.0000",0.2660,4.1693,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,4.0000,98.0000
num_real_estate_loans,"150,000.0000",1.0182,1.1298,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,2.0000,3.0000,4.0000,6.0000,54.0000
num_60_89_days_late,"150,000.0000",0.2404,4.1552,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,98.0000
num_dependents,"146,076.0000",0.7572,1.1151,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,4.0000,5.0000,20.0000


У людей с пропусками, судя по всему количество системных кодов в признаках с количеством просрочек гораздо выше. Проверим.

In [114]:
temp = data[data["num_dependents"].isna()]
temp["system_code_in_days_late"] = temp["num_30_59_days_late"].apply(lambda x: 1 if x == 96 or x == 98 else 0)
system_code_in_days_late_rate_in_nan_num_deps = temp["system_code_in_days_late"].mean()

In [115]:
temp = data[data["num_dependents"] == 0]
temp["system_code_in_days_late"] = temp["num_30_59_days_late"].apply(lambda x: 1 if x == 96 or x == 98 else 0)
system_code_in_days_late_rate_in_zero_num_deps = temp["system_code_in_days_late"].mean()

In [116]:
data["system_code_in_days_late"] = data["num_30_59_days_late"].apply(lambda x: 1 if x == 96 or x == 98 else 0)
system_code_in_days_late_rate_overall = data["system_code_in_days_late"].mean()

In [117]:
system_code_in_days_late_rate_in_nan_num_deps, system_code_in_days_late_rate_in_zero_num_deps, system_code_in_days_late_rate_overall

(np.float64(0.007900101936799185),
 np.float64(0.002163356424478148),
 np.float64(0.0017933333333333334))

Видно, что частота системных кодов в объектах с пропущенным `num_dependents` почти в 4 раза выше, чем в датасете в целом.

У объектов с пропусками в `num_dependents` отличаются распределения:
1. `revolving_utilization` - в среднем почти в 2 раза выше;
2. Количество системных кодов в 4 раза выше, чем в среднем по датасету;
3. `debt_ratio` - явно переходит к смыслу абсолюных значений;
4. `num_open_credit_lines` - в среднем в полтора раза ниже;
5. `num_real_estate_loans` - в среднем почти в 2 раза ниже;

Таким образом, заёмщики с пропущенным `num_dependents` в прошлом:
1. В среднем брали в 2 раза меньше кредитов;
2. количество системных кодов в просрочках в 4 раза выше;
3. отсутствующий `num_dependents` не совпадает по распределением с людьми без иждевенцев.

**bad rate:**

bad rate объектов с пропусками.

In [118]:
bad_rate_nan_num_dependents = round(data[data["num_dependents"].isna()]["target"].mean(), 4)

In [119]:
navigation["bad rate объектов с пустым num_dependents"] = "bad_rate_nan_num_dependents"

Доверительный интервал bad rate объектов с пропусками в num_dependents.

In [120]:
temp = data[data["num_dependents"].isna()]

ci_low_nan_num_deps_bad_rate, ci_high_nan_num_deps_bad_rate = proportion_confint(
    count=temp["target"].sum(),
    nobs=temp.shape[0],
    alpha=0.05,
    method="wilson"
)

ci_nan_num_deps_bad_rate = (round(ci_low_nan_num_deps_bad_rate, 4), round(ci_high_nan_num_deps_bad_rate, 4))

In [121]:
navigation["доверительный интервал bad rate объектов с пропусками num_dependents"] = "ci_nan_num_deps_bad_rate"

bad rate объектов не имеющих иждевенцев.

In [122]:
bad_rate_zero_num_dependents = round(data[data["num_dependents"] == 0]["target"].mean(), 4)

In [123]:
navigation["bad rate объектов с нулевым num_dependents"] = "bad_rate_zero_num_dependents"

Доверительный интервал bad rate объектов с нулевым num_dependents.

In [124]:
temp = data[data["num_dependents"] == 0]

ci_low_zero_num_deps_bad_rate, ci_high_zero_num_deps_bad_rate = proportion_confint(
    count=temp["target"].sum(),
    nobs=temp.shape[0],
    alpha=0.05,
    method="wilson"
)

ci_zero_num_deps_bad_rate = (round(ci_low_zero_num_deps_bad_rate, 4), round(ci_high_zero_num_deps_bad_rate, 4))

In [125]:
navigation["доверительный интервал bad rate объектов с нулевым num_dependents"] = "ci_zero_num_deps_bad_rate"

Сравнение ситуации по датасету в целом.

In [126]:
overall_bad_rate, ci_overall_bad_rate

(np.float64(0.0668), (0.0656, 0.0681))

In [127]:
bad_rate_nan_num_dependents, ci_nan_num_deps_bad_rate

(np.float64(0.0456), (0.0395, 0.0526))

In [128]:
bad_rate_zero_num_dependents, ci_zero_num_deps_bad_rate

(np.float64(0.0586), (0.0571, 0.0602))

Проверим значимость отличия.

In [129]:
mask = data["num_dependents"].isna()

make_z_test(data=data, mask=mask, alpha=0.05)

np.True_

Значимые отличия есть, вероятность дефолта в будущем у таких заёмщиков на 25 процентов меньше. Кроме того, люди без иждевенцев по bad rate не совпадают с объектами с пропусками в этом признаке.

**Вывод по пропускам в num_dependents:**
1. Люди с пропущенным `num_dependents` брали в среднем в 2 раза меньше кредитов и в среднем в 2-4 раза чаще сталкивались с короткими и длительными просрочками;
2. bad rate отличается значительно, но тем неменее по одному этому признаку делать выводы нельзя;
3. Стоит сделать флаг, который будет сигнализировать об отсутствии данных о количестве иждевенцев заёмщика.

In [130]:
quality_resume.loc[ quality_resume.shape[0] ] = [
    "missing num_dependents",
    "Нет никаких ограничений.",
    "1. количество кредитов в среднем в 2 раза ниже, чем по датасету в целом;\n"
    "2. количество краткосрочных просрочек в среднем в 2 раза выше, чем по датасету в целом;\n"
    "3. количество краткосрочных просрочек в среднем в 4 раза выше, чем по датасету в целом;\n"
    "4. bad rate в среднем на 25% ниже, чем по датасету в целом;\n"
    "5. объекты с пропуском в num_dependents не совпадают по распределением с объектами с нулевым количеством иждевенцев.",
    "1. Сделать флаг об отсутствии данных о количестве иждевенцев."
]

## Резюме

In [131]:
quality_resume.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

,feature,limitations,summary,actions
0,duplicate,Нет никаких ограничений.,"Дубликаты пусть и выглядят как пустные данные (что выражается в количественных характеристиках их представителей), однако сами объекты не обладают уникальными идентификаторами, а их большее количество в принципе делают не удивительными, дубликаты по всем значениям. Таким образом, вполне возможно, что это дубликаты не по объектам, а по значениям.",Ничего не предпринимать.
1,revolving_utilization,1. признак не может принимать отрицательных значений; 2. признак не должен принимать значения сильно выше 1; 3. положителельные значения признака должны сопровождаться положительными значениями `num_open_credit_lines`.,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. экстремально большие значения, вероятнее всего, объясняются сменой смысла признака: из отношения к абсолютным значениям; 3. в рамках текущей документации сделать вывод о ситуации, когда `revolving_utilization` положителен, а `num_open_credit_lines` = 0, не представляется возможным",1. сохранить исходное значение; отдельно отметить экстремальные значения; способ обработки определить после анализа их связи с риском.
2,age,1. признак не может принимать отрицательных значений; 2. признак не может принимать значения меньше 18 лет (возраст совершеннолетия); 3. признак не должен принимать значения сильно выше 90.,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. есть один представитель с возрастом 0 лет - это явно ошибка; 3. сделать вывод об ошибочности представителей с экстремальными значениями в рамках текущей документации невозможно, считаем их нормальными.",1. сделать флаг об ошибке в признаке (возраст ниже 18 лет).
3,num_30_59_days_late / num_60_89_days_late / num_90_days_late,"1. признаки не могут принимать отрицательных значений; 2. признаки не могут принимать слишком большие значения (порядка 50, сотен и так далее).","1. признаки не содержат ошибок, связанных с отрицательными значениями; 2. признаки не содержат ненормально больших значений, кроме 96 и 98; 3. значения 96 и 98 являются системными кодами.",1. сделать флаги о системных кодах 96 и 98.
4,debt_ratio,1. признак не может принимать отрицательных значений; 2. признак не должен принимать значения сильно выше 1.,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. экстремальные значения имеют распределение, несовместимое с буквальной интерпретацией показателя как обычного ratio. Возможными объяснениями являются особенности расчёта показателя при малом/нулевом знаменателе, изменение семантики поля или ошибки данных. Исходная документация не позволяет установить точную причину.",1. сохранить исходное значение; отдельно отметить экстремальные значения; способ обработки определить после анализа их связи с риском.
5,monthly_income,"1. признак не может принимать отрицательных значений; 2. признак не может принимать совсем больших значений, порядка десятков и сотен миллионов.","1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не содержит ошибок, связанных с экстремальными значениями.",Ничего не предпринимать.
6,num_open_credit_lines,1. признак не может принимать отрицательных значений; 2. признак не может принимать сильно больших значений (порядка сотен).,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не содержит ошибок, связанных с экстремальными значениями.",Ничего не предпринимать.
7,num_real_estate_loans,1. признак не может принимать отрицательных значений; 2. признак не может принимать сильно больших значений (порядка сотен).,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не содержит ошибок, связанных с экстремальными значениями.",Ничего не предпринимать.
8,num_dependents,1. признак не может принимать отрицательных значений; 2. признак не может принимать сильно больших значений (порядка сотен).,"1. признак не содержит ошибок, связанных с отрицательными значениями; 2. признак не 